# Readme Lory

Dieses Jupyter Notebook bereitet Datenobjekte und Metadaten aus dem Repository Lory vor für die digitale Langzeitarchivierung (DLZA) der ZHB Luzern, basierend auf der GOCFL implementierung von Jürgen Enge, https://github.com/je4/gocfl .

Um eine Collection korrekt vorzubereiten, sollten die einzelnen Scripts immer alle in der richtigen Reihenfolge ausgeführt werden. 

## 1 - Orderstruktur erstellen

Die Scripts in diesem Notebook basieren darauf, dass folgende Ordnerstruktur vorhanden ist, und dass die Ordner jeweils zu Beginn geleert werden. 

- Ordner 'files': hier werden diverse Dateien (z.B. eine Textdatei mit allen Signaturen) abgelegt
- Ordner 'info': hier werden die info.json Dateien abgelegt.
- Ordner 'metadata': hier werden die semantischen Metadaten zu den LZA-Objekten abgelegt
- Ordner 'objects': hier werden die zu archivierenden Datenobjekte (Payload) abgelegt. 

In [ ]:
import os
import shutil
from datetime import datetime

# remove existing directories
if os.path.exists('files'):    
    shutil.rmtree('files')

if os.path.exists('info'):
    shutil.rmtree('info')
    
if os.path.exists('metadata'):
    shutil.rmtree('metadata')

# exception: don't remove directory 'objects'
if os.path.exists('objects'):
    pass
else:
    os.mkdir('objects')
 
    
# create necessary directories:
os.mkdir('files')
os.mkdir('info')
os.mkdir('metadata')  

# list all directories in current working directory:
working_directory = './'
print(f"Working directory contains the following directories:")
items = os.listdir(working_directory)
for item in items:
    if os.path.isdir(item):
        print(item)
        
        
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))

## 2 - Erstellung der info.json

Dieses Script erstellt eine info.json Datei für alle Objekte der Collection und legt sie im Ordner 'info' als JSON Datei ab. 

### Basiskonfiguration mit Config.py

Beispieldaten für ZHB E-Rara. Anpassungen können in der config.py vorgenommen werden. 

    user = 'name of person ingesting this object'
    address = 'mailto:someone@internet.com'
    collection = 'ZHB Lory'
    collection_id = 'zhb_lory_XXX'
    ingest_workflow = 'zhb_lory'
    organisation = 'Zentral- und Hochschulbibliothek Luzern'
    organisation_id = 'zhb'
    signature = 'zhb_'
    
Die Lory-Signaturen setzen sich aus dem Zenodo-DOI zusammen. Im Config-File wird die Abteilung hinzugefügt, das Script ergänzt den DOI, die Community (set, keywords), z.B. unilu, phlu, hslu sowie einige weitere Metadaten.
    
### OAI Konfiguration 

Für die E-Rara-Bestände wird die Zenodo-OAI-Schnittstelle verwendet. Hier liegen sowohl die SIP als auch die Metadaten. Die Daten werden im Datacite-Format bezogen, da hier die vollständigsten Informationen vorliegen.

Zenodo: siehe https://developers.zenodo.org/#oai-pmh

    community: lory
    set name for OAI listrecords: user-lory

### Export info.json

Für jeden einzelnen record wird eine info.json-Datei erstellt im Format info/signature.json.
Das ganze Set wird am Ende noch als json- und Excel-Datei exportiert ins directory 'files' als menschenlesbarer Nachweis, welche Datenobjekte eingelagert wurden. Auch eine Liste aller Signaturen wird als Text-Datei dort abgelegt. 


In [ ]:
from sickle import Sickle
import re
import json
import config
import requests
import pandas as pd
from datetime import datetime

# Initialize the client by passing the base URL and fetch records from the OAI Set

base_url = 'https://zenodo.org/oai2d'
prefix = 'oai_dc'
set_name = 'user-lory'

sickle = Sickle(base_url)
records = sickle.ListRecords(metadataPrefix=prefix, set=set_name)
record = records.next()

completed_iterating = False
recordCount = 1

# other output variables

completeSet = []    
today = datetime.today().strftime('%Y-%m-%d %H:%M:%S')
now = datetime.now().isoformat()
sigfile = "files/signatures.txt"

# Iterate through records and collect metadata info for json export

# !!!! Uncomment one of the next 2 lines for testing/production:

#while not completed_iterating:
while recordCount < 11:
    try:
        
        print("record counter:",recordCount)
        infoSet = {}
        
        infoSet["additional"] = ''
        infoSet["address"] = config.address
        infoSet["collection"] = config.collection
        infoSet["collection_id"] = config.collection_id
        infoSet["created"] = now
        infoSet["identifiers"] = record.metadata['identifier']
        infoSet["ingest_workflow"] = config.ingest_workflow
        infoSet["keywords"] = config.keywords
        infoSet["last_changed"] = now
        infoSet["organisation"] = config.organisation
        infoSet["organisation_id"] = config.organisation_id
        infoSet["references"] = []
        infoSet["sets"] = []
        infoSet["signature"] = ''
        infoSet["title"] = record.metadata["title"][0]
        infoSet["user"] = config.user                
        
        # add Lory communities (OAI-Sets) to sets:
        infoSet["sets"].extend(config.sets)
        oai_sets = record.header.setSpecs
        infoSet["sets"].extend(oai_sets)
        
        # extract sets and other dois
        relations = record.metadata["relation"]

        for i in relations:
            if i.startswith('doi:'):
                infoSet["identifiers"].append(i)
    
        # extract signature, url 
        identifiers = record.metadata["identifier"]
        
        for i in identifiers:
            if i.startswith('https://zenodo'):
                infoSet["references"].append(i)
                infoSet["identifiers"].remove(i)
                
            if i.startswith('doi:10.5281/zenodo'):
                zenodo_id = i[19:]
                infoSet["identifiers"].append('zenodo:'+zenodo_id)
                foldername = i[4:].replace('.','_').replace('/','_') 
                signature = config.signature+foldername
                infoSet["additional"] = foldername
                infoSet["signature"] = signature
                print("Signature:",signature)

        print(infoSet)             
        
        # prepare filename for json export
        info_json = json.dumps(infoSet, indent=4, ensure_ascii=False)
        infofile = f"info/{signature}.json"
        with open(infofile, "w") as outfile:
            outfile.write(info_json)
            print(f"---\ninfo.json saved as {infofile}")
        
        # add info to completeSet
        completeSet.append(infoSet)             
        
        
        # append signature to signatures.txt
        with open(sigfile, 'a') as file:
            file.write(signature)
            file.write("\n")
            print(f"Signature {signature} appended to {sigfile}\n")
        

        #continue with next record
        recordCount = recordCount +1
        record = records.next()
        
    except StopIteration:
        completed_iterating = True

# Writing completeSet as json file
fulldump = json.dumps(completeSet, indent=4, ensure_ascii=False)
fulljsonfile = "files/lory_complete_set.json"
with open(fulljsonfile, "w") as outfile:
    outfile.write(fulldump)
    print(f"---\nJSON File for entire collection written to {fulljsonfile}")
    
# Writing completeSet as Excel file
fullexcelfile = "files/lory_complete_set.xlsx"
df_json = pd.read_json(fulljsonfile)
df_json.to_excel(fullexcelfile)
print(f"Excel File for entire collection written to {fullexcelfile}")
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))


## 3 - Semantische Metadaten

Via Zenodo können derzeit folgende Metadatenformate extrahiert werden (über OAI):
https://www.zenodo.org/oai2d?verb=ListMetadataFormats

Aus https://developers.zenodo.org/#metadata-formats:

We recommend you harvest using the “oai_datacite” metadata format. The format contains the most complete metadata and is our primary supported export format. 

### Datacite Metadaten aus Zenodo

Mit dem Zenodo-Identifier werden die Metadaten via OAI aus Zenodo extrahiert und abgespeichert unter metadata/{signature}/signature.xml. Für gocfl create müssen die Metadaten pro Objekt in einem eigenen Ordner liegen. 

Zusätzlich könnten diesselben Metadaten auch im DC-Format heruntergeladen werden. 
prefix = oai_dc

### Delay (rate limiting)

Global limit for guest users: 60 requests per minute, 2000 requests per hour
OAI-PMH API harvesting: 120 requests per minute
=> pro loop 1 Sekunde sleep() einbauen. 

In [ ]:
from sickle import Sickle
import json
import os
from datetime import datetime
import time

sickle = Sickle('https://zenodo.org/oai2d')
file_name = 'files/lory_complete_set.json'
counter = 1
with open(file_name) as data_file:    
    data = json.load(data_file)
    for value in data:
        
        # find necessary values
        zenodo_id = value["identifiers"][1]
        foldername = value["additional"]
        signature = value["signature"]
        
        md_path = f'metadata/{foldername}'
        # make a directory for each object        
        if os.path.exists(md_path):
            pass
        else:
            os.mkdir(md_path)
            
        datacite_response = sickle.GetRecord(identifier=zenodo_id, metadataPrefix='oai_datacite')

        datacite_md_file = f'{md_path}/{foldername}_datacite.xml'
        
        with open(datacite_md_file, 'w') as fp:
            fp.write(datacite_response.raw)
            
        print(f'{signature}: Metadata downloaded. Total downloads: {counter}')
        
        # wait 1 second so as not to overshoot zenodo rate limiting. 
        time.sleep(1)
        counter = counter + 1
        
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))


## 4 - Abholung Datenobjekte 

Für gocfl wird vorausgesetzt, dass im Ordner 'objects' ein Unterordner mit dem DOI existiert, worin die Datei liegt (in wenigen Fällen auch mehrere Dateien). 

Der DOI / Foldername ist in dieser Form (mit Unterstrichen) im Feld "additional" in der JSON-Datei gespeichert. 



In [6]:
import requests
import json
import os
from zipfile import ZipFile
from datetime import datetime
import config

file_name = 'files/lory_complete_set.json'
counter = 1
download_manually = 'objects/download_manually.txt'

ACCESS_TOKEN = config.access_token

with open(file_name) as data_file:    
    data = json.load(data_file)
    for value in data:
        # find necessary values
        zenodo_id = value["identifiers"][-1][7:]
        print(counter,"\nDownloading files from:",zenodo_id)
        counter = counter+1
        foldername = value["additional"]
        
        # make a directory for each object
        folderpath = f'objects/{foldername}'
        if os.path.exists(folderpath):
            pass
        else:
            os.mkdir(folderpath)
                  
        # get the file download link from zenodo:        
        zenodo_link = f'https://zenodo.org/api/records/{zenodo_id}'        
        print(zenodo_link)
        
        response = requests.get(zenodo_link, params={'access_token': ACCESS_TOKEN})
        zenodo_object = response.json()
        #print(zenodo_object.keys())
        
        try: 
            # if exists files
            #print(zenodo_object['files'])
            for file in zenodo_object['files']:
                download_url = file['links']['self']
                #file_name = download_url.split("/")[-1]
                file_name = file['key']

                print("filename:",file_name)
                local_file = f'{folderpath}/{file_name}'

                # distinguish between pdf files and other files

                if file['type'] == 'pdf':
                    response = requests.get(download_url)
                    with open(local_file, mode="wb") as file:
                        file.write(response.content)

                else:
                    print("Not a PDF, download manually:")
                    # append download url to download_manually.txt
                    with open(download_manually, 'a') as file:
                        file.write(download_url)
                        file.write("\n")
                        print(f"URL {download_url} appended to {download_manually}\n")
        except KeyError:
            print("KeyError: files not found, download manually:")
            # append download url to download_manually.txt
            with open(download_manually, 'a') as file:
                file.write(zenodo_link)
                file.write("\n")
                print(f"URL {zenodo_link} appended to {download_manually}\n")
            

print("\nFinished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))

1 
https://zenodo.org/api/records/804333
filename: 2010 mobility analysis for the three Swiss cities in the alpine space in Switzerland.pdf
2 
https://zenodo.org/api/records/7382807
KeyError: files not found, download manually:
URL https://zenodo.org/api/records/7382807 appended to objects/download_manually.txt

3 
https://zenodo.org/api/records/1402069
KeyError: files not found, download manually:
URL https://zenodo.org/api/records/1402069 appended to objects/download_manually.txt

4 
https://zenodo.org/api/records/1320751
KeyError: files not found, download manually:
URL https://zenodo.org/api/records/1320751 appended to objects/download_manually.txt

5 
https://zenodo.org/api/records/5470914
filename: Gesamtliste_Veroffentlichungen_M._Bauman_8-2021.pdf
6 
https://zenodo.org/api/records/3265645
filename: feinschwarz.net-70_Jahre_Allgemeine_Erklarung_der_Menschenrechte.pdf
7 
https://zenodo.org/api/records/7215691
filename: Schulze_Waltenspuel_Follow_the_Films.pdf
8 
https://zenodo.or